In [4]:
import sys
print(sys.executable)

from tavily import TavilyClient

print("working")

d:\work_project\persanol_project\SocialAgent-Graph\.venv\Scripts\python.exe
working


In [1]:
! pip install praw   

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
import json

from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.4
)

In [5]:
tavily = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

In [6]:
def research_topic(topic: str):

    results = tavily.search(
        query=topic,
        search_depth="advanced",
        max_results=5
    )

    research_data = []

    for item in results["results"]:
        research_data.append({
            "title": item.get("title"),
            "url": item.get("url"),
            "content": item.get("content")
        })

    return research_data

In [6]:
topic = "best way to learn LangGraph for production AI agents"
platform = "reddit"

research = research_topic(topic)

research[:2]

[{'title': 'AI Agents with LangChain and LangGraph | Online Course | Udacity',
  'url': 'https://www.udacity.com/course/ai-agents-with-langchain-and-langgraph--cd13764',
  'content': 'Learn to implement flexible, reliable AI agents using LangGraph by designing agentic workflows, managing parallel updates with reducers, and customizing behavior via configuration.\n25. LangGraph Database Agents\n\n    Learn to build LangGraph database agents that use AI to convert natural language into SQL queries, enabling users to query databases conversationally without writing SQL.\n26. LangGraph Messaging\n\n    Learn techniques in LangGraph for efficient message handling—limiting, trimming, summarizing messages, and using multiple state schemas for safe, modular agentic workflows.\n27. Short-Term Agent Memory [...] Learn to build a simple LangChain agent that combines LLMs, memory, and tools for automated, multi-step workflows and natural user interactions.\n22. Agentic Workflows with LangGraph\n\n

In [7]:
def evaluate_research(topic: str, research_data: list) -> str:
    research_text = json.dumps(research_data, indent=2)

    prompt = f"""
Evaluate this research.

Topic:
{topic}

Research:
{research_text}

Check:
- Is it relevant?
- What are the key useful points?
- What is missing?
- What should we be careful not to overclaim?

Return concise notes.
"""

    response = llm.invoke(prompt)
    return response.content

In [8]:
evaluation = evaluate_research(topic, research)

print(evaluation)

**Relevance:**  
All sources are highly relevant, focusing specifically on learning and applying LangGraph for building production-ready AI agents. They cover conceptual foundations, practical tutorials, workflows, and production considerations.

**Key Useful Points:**  
- **Core Concepts:** LangGraph revolves around three pillars: Tools (Python functions wrapped as callable tools), State (agent memory/state management), and Graph (nodes and edges representing workflows and decision logic).  
- **Workflow Design:** Building agentic workflows using nodes (LLM/tool calls), edges (conditional routing), and state updates enables modular, adaptive AI agents.  
- **Practical Implementation:** Hands-on tutorials show how to register nodes, define edges, set entry points, and implement conditional logic to build dynamic agents.  
- **Production Readiness:** Emphasis on error handling, retries, observability, metrics, human-in-the-loop control, and ongoing monitoring for reliable, scalable AI a

In [8]:
def generate_social_response(
    topic: str,
    platform: str,
    research_data: list,
    evaluation: str
) -> str:
    research_text = json.dumps(research_data, indent=2)

    prompt = f"""
Write a helpful {platform} response.

Topic:
{topic}

Research:
{research_text}

Evaluation:
{evaluation}

Rules:
- Sound natural, like a real Reddit user.
- Be helpful and specific.
- Do not sound promotional.
- Do not overclaim.
- Mention practical steps.
- Keep under 250 words.
"""

    response = llm.invoke(prompt)
    return response.content

In [10]:
social_response = generate_social_response(
    topic=topic,
    platform=platform,
    research_data=research,
    evaluation=evaluation
)

print(social_response)

If you want to learn LangGraph for building production AI agents, the best approach is a mix of hands-on experimentation and understanding its core concepts: Tools, State, and Graph.

Start by getting comfortable with how LangGraph models workflows as graphs—nodes represent LLM or tool calls, edges handle conditional routing, and state keeps track of info as your agent runs. The free tutorials and articles (like the ones on freeCodeCamp and Towards Data Science) walk you through setting up simple workflows, registering nodes, defining edges, and managing state with TypedDict or Pydantic.

I’d recommend building a small project first—maybe an agent that queries a database or calls an external API—so you can see how to wire nodes and edges together, handle errors, and update state. This hands-on step is crucial because LangGraph strikes a balance between giving you control and abstracting complexity, so you want to understand how your logic flows.

Once you’re comfortable, focus on produ

In [9]:
def quality_check(response: str):

    prompt = f"""
You are a strict senior social media reviewer.

Review this response and score it from 1 to 10.

Scoring:
- 9-10 = excellent, ready to post
- 7-8 = good, minor edits needed
- 5-6 = average, needs improvement
- 1-4 = poor, do not post

Rules:
- approved must be true only if score >= 8
- approved must be false if score < 8
- score must be an integer from 1 to 10

Response:
{response}

Return ONLY valid JSON:

{{
  "score": 8,
  "approved": true,
  "issues": ["issue 1"],
  "improved_response": "final improved version"
}}
"""

    result = llm.invoke(prompt).content
    return json.loads(result)

In [14]:
qc = quality_check(social_response)

qc

{'score': 8,
 'approved': True,
 'issues': ['The response could benefit from slight restructuring for clarity and conciseness, and the mention of specific tutorial sources could include direct links or clearer references.'],
 'improved_response': 'If you want to learn LangGraph for building production AI agents, the best approach combines hands-on experimentation with understanding its core concepts: Tools, State, and Graph.\n\nStart by getting comfortable with how LangGraph models workflows as graphs—nodes represent LLM or tool calls, edges handle conditional routing, and state keeps track of information as your agent runs. Free tutorials and articles, such as those on freeCodeCamp and Towards Data Science, guide you through setting up simple workflows, registering nodes, defining edges, and managing state with TypedDict or Pydantic.\n\nI recommend building a small project first—perhaps an agent that queries a database or calls an external API—so you can see how to wire nodes and edge

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [16]:
class SocialAgentState(TypedDict):
    topic: str
    platform: str
    research: list
    evaluation: str
    social_response: str
    quality_score: int
    approved: bool
    final_response: str

In [19]:
def research_node(state: SocialAgentState):
    research = research_topic(state["topic"])
    return {"research": research}


def evaluation_node(state: SocialAgentState):
    evaluation = evaluate_research(
        topic=state["topic"],
        research_data=state["research"]
    )
    return {"evaluation": evaluation}


def generate_response_node(state: SocialAgentState):
    social_response = generate_social_response(
        topic=state["topic"],
        platform=state["platform"],
        research_data=state["research"],
        evaluation=state["evaluation"]
    )
    return {"social_response": social_response}


def quality_check_node(state: SocialAgentState):
    qc = quality_check(state["social_response"])

    return {
        "quality_score": qc["score"],
        "approved": qc["approved"],
        "final_response": qc["improved_response"]
    }

In [18]:
builder = StateGraph(SocialAgentState)

builder.add_node("research", research_node)
builder.add_node("evaluation", evaluation_node)
builder.add_node("generate_response", generate_response_node)
builder.add_node("quality_check", quality_check_node)

builder.add_edge(START, "research")
builder.add_edge("research", "evaluation")
builder.add_edge("evaluation", "generate_response")
builder.add_edge("generate_response", "quality_check")
builder.add_edge("quality_check", END)

graph = builder.compile()

In [19]:
result = graph.invoke({
    "topic": "best way to learn LangGraph for production AI agents",
    "platform": "reddit"
})

result

{'topic': 'best way to learn LangGraph for production AI agents',
 'platform': 'reddit',
 'research': [{'title': 'AI Agents with LangChain and LangGraph | Online Course | Udacity',
   'url': 'https://www.udacity.com/course/ai-agents-with-langchain-and-langgraph--cd13764',
   'content': 'Learn to implement flexible, reliable AI agents using LangGraph by designing agentic workflows, managing parallel updates with reducers, and customizing behavior via configuration.\n25. LangGraph Database Agents\n\n    Learn to build LangGraph database agents that use AI to convert natural language into SQL queries, enabling users to query databases conversationally without writing SQL.\n26. LangGraph Messaging\n\n    Learn techniques in LangGraph for efficient message handling—limiting, trimming, summarizing messages, and using multiple state schemas for safe, modular agentic workflows.\n27. Short-Term Agent Memory [...] Learn to build a simple LangChain agent that combines LLMs, memory, and tools for 

In [20]:
print("Score:", result["quality_score"])
print("Approved:", result["approved"])
print("\nFinal Response:\n")
print(result["final_response"])

Score: 8
Approved: True

Final Response:

If you’re aiming to learn LangGraph for building production AI agents, I recommend a hands-on approach focused on its three core concepts: **Tools**, **State**, and **Graph**. Start by understanding how to wrap Python functions as tools (using decorators like `@tool`), since these are the building blocks your agents will call.

Next, get comfortable managing the agent’s **state**—this is the memory or context your agent maintains as it runs. LangGraph supports simple schemas with TypedDict or more robust validation with Pydantic, helping keep your workflows reliable.

Then, dive into building workflows as graphs: nodes represent LLM or tool calls, and edges define conditional logic and routing. Try creating simple workflows yourself—like intent classification followed by different paths—so you grasp how state flows and decisions are made.

For production readiness, don’t overlook error handling, retries, and observability. LangGraph supports hu

In [21]:
def route_quality(state: SocialAgentState):
    if state["quality_score"] >= 8:
        return "approved"
    return "retry"

In [22]:
builder = StateGraph(SocialAgentState)

builder.add_node("research", research_node)
builder.add_node("evaluation", evaluation_node)
builder.add_node("generate_response", generate_response_node)
builder.add_node("quality_check", quality_check_node)

builder.add_edge(START, "research")
builder.add_edge("research", "evaluation")
builder.add_edge("evaluation", "generate_response")
builder.add_edge("generate_response", "quality_check")

builder.add_conditional_edges(
    "quality_check",
    route_quality,
    {
        "approved": END,
        "retry": "generate_response"
    }
)

graph = builder.compile()

In [23]:
result = graph.invoke({
    "topic": "best way to learn LangGraph for production AI agents",
    "platform": "reddit"
})

print("Score:", result["quality_score"])
print("Approved:", result["approved"])
print(result["final_response"])

Score: 8
Approved: True
If you want to learn LangGraph—a framework for building production-ready AI agents—I recommend a hands-on approach combined with solid conceptual grounding.

Start by understanding the three core components: **Tools** (Python functions wrapped as callable tools), **State** (using TypedDict or Pydantic schemas to keep your agent’s memory reliable), and the **Graph** itself (nodes representing LLM calls or tools, edges defining conditional flows). This mental model helps you design modular, adaptive workflows.

Begin by building simple workflows, such as a basic intent classifier that routes to different nodes based on user input. This will help you get comfortable with adding nodes and edges programmatically and setting entry points. Experiment with conditional edges and state updates to deepen your understanding.

Once comfortable, focus on production concerns: add error handling, retries, and observability features like logging and metrics. Incorporate human-in

In [24]:
class SocialAgentState(TypedDict):
    topic: str
    platform: str
    research: list
    evaluation: str
    social_response: str
    quality_score: int
    approved: bool
    final_response: str
    retry_count: int

In [25]:
def generate_response_node(state: SocialAgentState):
    social_response = generate_social_response(
        topic=state["topic"],
        platform=state["platform"],
        research_data=state["research"],
        evaluation=state["evaluation"]
    )

    retry_count = state.get("retry_count", 0) + 1

    return {
        "social_response": social_response,
        "retry_count": retry_count
    }

In [26]:
def route_quality(state: SocialAgentState):
    if state["quality_score"] >= 8:
        return "approved"

    if state["retry_count"] >= 3:
        return "stop"

    return "retry"

In [27]:
builder = StateGraph(SocialAgentState)

builder.add_node("research", research_node)
builder.add_node("evaluation", evaluation_node)
builder.add_node("generate_response", generate_response_node)
builder.add_node("quality_check", quality_check_node)

builder.add_edge(START, "research")
builder.add_edge("research", "evaluation")
builder.add_edge("evaluation", "generate_response")
builder.add_edge("generate_response", "quality_check")

builder.add_conditional_edges(
    "quality_check",
    route_quality,
    {
        "approved": END,
        "retry": "generate_response",
        "stop": END
    }
)

graph = builder.compile()

In [28]:
result = graph.invoke({
    "topic": "best way to learn LangGraph for production AI agents",
    "platform": "reddit",
    "retry_count": 0
})

print("Score:", result["quality_score"])
print("Approved:", result["approved"])
print("Retries:", result["retry_count"])
print(result["final_response"])

Score: 8
Approved: True
Retries: 1
If you’re looking to learn LangGraph for building production-ready AI agents, here’s what helped me get up to speed:

1. **Understand the core concepts first:** LangGraph revolves around three pillars — *Tools* (Python functions wrapped as callable tools), *State* (agent memory and context, usually managed with TypedDict or Pydantic), and *Graph* (nodes and edges defining your workflow). Grasping these basics will make everything else clearer.

2. **Start small and build workflows:** Try implementing simple workflows where nodes represent LLM calls or tool invocations, and edges handle conditional routing. This hands-on approach is key. Tutorials available on platforms like freeCodeCamp or Medium provide step-by-step guidance for creating agents.

3. **Use decorators to create tools easily:** LangGraph allows you to convert Python functions into tools using decorators, simplifying integration and testing by reducing boilerplate code.

4. **Focus on st

In [29]:
class SocialAgentState(TypedDict):
    topic: str
    platform: str
    research: list
    evaluation: str
    social_response: str
    quality_score: int
    approved: bool
    final_response: str
    retry_count: int
    human_approved: bool

In [16]:
def human_approval_node(state: SocialAgentState):
    print("\nFinal response:\n")
    print(state["final_response"])

    user_input = input("\nApprove this response? yes/no: ").strip().lower()

    return {
        "human_approved": user_input == "yes"
    }

In [17]:
def route_quality(state: SocialAgentState):
    if state["quality_score"] >= 8:
        return "human_approval"

    if state["retry_count"] >= 3:
        return "stop"

    return "retry"

In [18]:
def route_human_approval(state: SocialAgentState):
    if state["human_approved"]:
        return "approved"
    return "stop"

In [33]:
builder = StateGraph(SocialAgentState)

builder.add_node("research", research_node)
builder.add_node("evaluation", evaluation_node)
builder.add_node("generate_response", generate_response_node)
builder.add_node("quality_check", quality_check_node)
builder.add_node("human_approval", human_approval_node)

builder.add_edge(START, "research")
builder.add_edge("research", "evaluation")
builder.add_edge("evaluation", "generate_response")
builder.add_edge("generate_response", "quality_check")

builder.add_conditional_edges(
    "quality_check",
    route_quality,
    {
        "human_approval": "human_approval",
        "retry": "generate_response",
        "stop": END
    }
)

builder.add_conditional_edges(
    "human_approval",
    route_human_approval,
    {
        "approved": END,
        "stop": END
    }
)

graph = builder.compile()

In [34]:
result = graph.invoke({
    "topic": "best way to learn LangGraph for production AI agents",
    "platform": "reddit",
    "retry_count": 0,
    "human_approved": False
})

print(result["human_approved"])


Final response:

If you’re looking to learn LangGraph for building production-ready AI agents, I’d recommend starting with the core concepts: **Tools, State, and Graph**. Think of tools as Python functions your agent can call (easy to create with decorators), state as the agent’s memory (using TypedDict or Pydantic schemas), and the graph as your workflow—nodes are LLM or tool calls, edges handle conditional logic and routing.

The best way to get comfortable is by **building simple workflows yourself**. For example, create a small agent that classifies user intent, calls a tool based on that intent, and updates state accordingly. This hands-on approach helps you understand how LangGraph balances control and abstraction without hiding too much complexity.

Also, pay attention to LangGraph’s built-in features like error handling, retries, and observability tools. These are crucial for production reliability but don’t assume they solve all operational challenges—you’ll still need to des

In [12]:
class SocialAgentState(TypedDict):
    topic: str
    platform: str
    research: list
    evaluation: str
    social_response: str
    quality_score: int
    approved: bool
    final_response: str
    retry_count: int
    human_approved: bool
    posted: bool
    posted_url: str

In [13]:
import praw
import os

def social_upload_node(state: SocialAgentState):
    if not state["human_approved"]:
        return {
            "posted": False,
            "posted_url": None
        }

    reddit = praw.Reddit(
        client_id=os.getenv("REDDIT_CLIENT_ID"),
        client_secret=os.getenv("REDDIT_CLIENT_SECRET"),
        username=os.getenv("REDDIT_USERNAME"),
        password=os.getenv("REDDIT_PASSWORD"),
        user_agent=os.getenv("REDDIT_USER_AGENT"),
    )

    subreddit = reddit.subreddit("test")

    submission = subreddit.submit(
        title=f"AI Discussion: {state['topic'][:180]}",
        selftext=state["final_response"]
    )

    return {
        "posted": True,
        "posted_url": f"https://reddit.com{submission.permalink}"
    }

In [22]:
def route_human_approval(state: SocialAgentState):
    if state["human_approved"]:
        return "upload"
    return "stop"

In [23]:
builder = StateGraph(SocialAgentState)

builder.add_node("research", research_node)
builder.add_node("evaluation", evaluation_node)
builder.add_node("generate_response", generate_response_node)
builder.add_node("quality_check", quality_check_node)
builder.add_node("human_approval", human_approval_node)
builder.add_node("social_upload", social_upload_node)

builder.add_edge(START, "research")
builder.add_edge("research", "evaluation")
builder.add_edge("evaluation", "generate_response")
builder.add_edge("generate_response", "quality_check")

builder.add_conditional_edges(
    "quality_check",
    route_quality,
    {
        "human_approval": "human_approval",
        "retry": "generate_response",
        "stop": END
    }
)

builder.add_conditional_edges(
    "human_approval",
    route_human_approval,
    {
        "upload": "social_upload",
        "stop": END
    }
)

builder.add_edge("social_upload", END)

graph = builder.compile()

In [24]:
result = graph.invoke({
    "topic": "best way to learn LangGraph for production AI agents",
    "platform": "reddit",
    "retry_count": 0,
    "human_approved": False
})

print("Human approved:", result["human_approved"])
print("Posted:", result.get("posted"))
print("Posted URL:", result.get("posted_url"))


Final response:

If you’re aiming to learn LangGraph for building production AI agents, here’s a practical approach that worked well for me and many others:

1. **Start with the basics:** Get comfortable with LangGraph’s core concepts—nodes (LLM calls, tool invocations), edges (conditional logic), and state management. The LangChain Academy’s free “Introduction to LangGraph” course (available on their official site) is a great starting point.

2. **Follow a structured learning path:** The Facebook Tech Titans group (a community of AI practitioners) shared a solid progression—from basic agent workflows, to debugging/monitoring production systems, and finally to advanced RAG pipelines. This staged approach helps you build confidence before tackling complex multi-agent setups.

3. **Hands-on experimentation:** Don’t just read—build simple workflows yourself. For example, create a small agent that classifies intents and routes requests conditionally. Medium and Towards Data Science have p